In [ ]:
import numpy as np
from pathlib import Path


def load_dataset(path="/home/gmarihuan/processed/chbmit_windows_all.npz"):
    """Return the whole dataset as X (float32), y, patient."""
    d = np.load(path, allow_pickle=True)
    X = d["X"].astype(np.float32)
    y = d["y"]
    patient = d["patient_id"]
    return X, y, patient


X, y, patient = load_dataset()

In [2]:
"""
10-fold patient-grouped cross-validation split for the CHB-MIT windows,
computed entirely in memory (nothing is written to disk).

Every patient ends up in exactly one test fold, so no patient's windows appear
in both train and test. StratifiedGroupKFold keeps the share of seizure windows
roughly similar across folds (plain GroupKFold is available too).

Usage from another script:
    from make_group_cv_split import load_dataset, get_cv_splits
    X, y, patient = load_dataset()
    for k, tr, te in get_cv_splits(y, patient):
        X_tr, y_tr, X_te, y_te = X[tr], y[tr], X[te], y[te]
"""
import numpy as np
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold

DATA_PATH = "/home/gmarihuan/processed/chbmit_windows_all.npz"
N_SPLITS = 10
SEED = 42
STRATIFIED = True

# In CHB-MIT, chb21 is the same subject as chb01 (recorded 1.5 years later).
# Grouping them together prevents subject leakage across folds.
GROUP_ALIASES = {"chb21": "chb01"}


def load_dataset(path=DATA_PATH):
    """Return the whole dataset as X (float32), y, patient."""
    d = np.load(path, allow_pickle=True)
    X = d["X"].astype(np.float32)
    y = d["y"]
    patient = d["patient_id"]
    return X, y, patient


def make_groups(patient):
    """Map patient ids to subject groups, merging aliased recordings."""
    pat = np.asarray(patient).astype(str)
    return np.array([GROUP_ALIASES.get(p, p) for p in pat])


def make_splits(y, groups, n_splits=N_SPLITS, seed=SEED, stratified=STRATIFIED):
    n_groups = len(np.unique(groups))
    if n_groups < n_splits:
        raise ValueError(f"Only {n_groups} groups, cannot make {n_splits} folds.")

    dummy_X = np.zeros(len(y), dtype=np.uint8)
    y_bin = (np.asarray(y) > 0).astype(int)  # seizure vs non-seizure for stratification

    if stratified:
        cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    else:
        cv = GroupKFold(n_splits=n_splits)
    return list(cv.split(dummy_X, y_bin, groups))


def check_and_report(splits, y, groups, verbose=True):
    """Verify there is no patient leakage and every window is tested exactly once."""
    n = len(y)
    fold_of = np.full(n, -1, dtype=np.int8)
    y_bin = (np.asarray(y) > 0).astype(int)

    if verbose:
        print(f"{'fold':>4} {'n_train':>9} {'n_test':>8} {'test_pos%':>9}  test patients")
    for k, (tr, te) in enumerate(splits):
        tr_g, te_g = set(groups[tr]), set(groups[te])
        assert tr_g.isdisjoint(te_g), f"Fold {k}: patient leakage {tr_g & te_g}"
        assert np.all(fold_of[te] == -1), f"Fold {k}: window in multiple test folds"
        fold_of[te] = k
        if verbose:
            print(f"{k:>4} {len(tr):>9} {len(te):>8} {100 * y_bin[te].mean():>8.2f}%  "
                  f"{sorted(te_g)}")

    assert np.all(fold_of >= 0), "Some windows were never in a test fold"
    if verbose:
        print(f"\nOverall positive rate: {100 * y_bin.mean():.2f}%  |  "
              f"windows: {n}  |  groups: {len(np.unique(groups))}")
    return fold_of


def get_cv_splits(y, patient, n_splits=N_SPLITS, seed=SEED,
                  stratified=STRATIFIED, verbose=True):
    """Return a list of (fold, train_idx, test_idx), computed in memory."""
    groups = make_groups(patient)
    splits = make_splits(y, groups, n_splits, seed, stratified)
    check_and_report(splits, y, groups, verbose)
    return [(k, tr, te) for k, (tr, te) in enumerate(splits)]

In [ ]:
X, y, patient = load_dataset()
folds = get_cv_splits(y, patient)

for k, tr, te in folds:
    X_tr, y_tr = X[tr], y[tr]
    X_te, y_te = X[te], y[te]


        # train / evaluate your model here

In [5]:
"""
AsymSETNet (grouped-spatial variant)
====================================

Same pipeline as the original AsymSETNet, but Spatial Module 1 no longer uses
a single (Cmax x 4) kernel over all 21 electrodes. Instead, every bipolar
electrode group (montage chain) has its OWN asymmetric conv block whose
kernel height equals the size of that group:

    left_temporal        4 ch  -> kernel (4, 4)
    left_parasagittal    4 ch  -> kernel (4, 4)
    right_parasagittal   4 ch  -> kernel (4, 4)
    right_temporal       4 ch  -> kernel (4, 4)
    midline              2 ch  -> kernel (2, 4)
    transverse_temporal  3 ch  -> kernel (3, 4)

Each group block collapses its electrodes to height 1. The G group outputs are
stacked along the height axis -> (B*N, f1, G, T'), and Spatial Module 2 fuses
the groups with a (G x 4) kernel (cross-region / inter-hemispheric features).

Pipeline:
    Raw EEG -> N segments
            -> Group Spatial Module 1 (one Asym Conv + SE per electrode group)
            -> stack groups -> Spatial Module 2 (G x 4 fusion, + SE)
            -> Global Average Pooling -> Temporal Module (stacked TCN)
            -> Classification Head (FC) -> Seizure / Normal

Channel order assumed (index in X[:, idx, :]), from REQUIRED_CHANNELS:
     0 FP1-F7   1 F7-T7    2 P7-O1    3 FP1-F3   4 F3-C3    5 C3-P3
     6 P3-O1    7 FP2-F4   8 F4-C4    9 C4-P4   10 P4-O2   11 FP2-F8
    12 F8-T8   13 T8-P8   14 P8-O2   15 FZ-CZ   16 CZ-PZ   17 P7-T7
    18 T7-FT9  19 FT9-FT10 20 FT10-T8
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# --------------------------------------------------------------------------- #
# 0. Electrode groups (indices into the 21-channel window)
# --------------------------------------------------------------------------- #
ELECTRODE_GROUPS = {
    "left_temporal":       [0, 1, 17, 2],    # FP1-F7, F7-T7, P7-T7, P7-O1
    "left_parasagittal":   [3, 4, 5, 6],     # FP1-F3, F3-C3, C3-P3, P3-O1
    "right_parasagittal":  [7, 8, 9, 10],    # FP2-F4, F4-C4, C4-P4, P4-O2
    "right_temporal":      [11, 12, 13, 14], # FP2-F8, F8-T8, T8-P8, P8-O2
    "midline":             [15, 16],         # FZ-CZ, CZ-PZ
    "transverse_temporal": [18, 19, 20],     # T7-FT9, FT9-FT10, FT10-T8
}

# P7-T7 is the reversed-polarity version of T7-P7 (the usual 3rd link of the
# left temporal chain). Channels listed here are multiplied by -1 so the left
# temporal chain is continuous: FP1-F7 -> F7-T7 -> T7-P7 -> P7-O1.
FLIP_CHANNELS = [17]


# --------------------------------------------------------------------------- #
# 1. Squeeze-and-Excitation block (unchanged)
# --------------------------------------------------------------------------- #
class SqueezeExcitation2d(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excite = nn.Sequential(
            nn.Linear(channels, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.squeeze(x).view(b, c)
        e = self.excite(s).view(b, c, 1, 1)
        return x * e


# --------------------------------------------------------------------------- #
# 2. Asymmetric convolutional block (unchanged)
# --------------------------------------------------------------------------- #
class AsymmetricConvBlock(nn.Module):
    """Conv (kh x kw, dilated) -> AvgPool(time) -> SE -> residual -> LayerNorm.
    kh == input height collapses the height axis to 1."""

    def __init__(self, in_channels, out_channels, kernel_size, dilation,
                 pool_size=(1, 2), se_reduction=16):
        super().__init__()
        kh, kw = kernel_size
        dh, dw = dilation
        pad_w = ((kw - 1) * dw) // 2

        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size=(kh, kw),
            dilation=(dh, dw), padding=(0, pad_w),
        )
        self.pool = nn.AvgPool2d(kernel_size=pool_size, ceil_mode=True)
        self.se = SqueezeExcitation2d(out_channels, reduction=se_reduction)

        self.skip_proj = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(kh, 1),
                      stride=(kh, 1)),
            nn.AvgPool2d(kernel_size=pool_size, ceil_mode=True),
        )
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        skip = self.skip_proj(x)

        y = self.conv(x)
        y = self.pool(y)
        y = self.se(y)

        if y.shape[-1] != skip.shape[-1]:
            w = min(y.shape[-1], skip.shape[-1])
            y, skip = y[..., :w], skip[..., :w]
        if y.shape[-2] != skip.shape[-2]:
            h = min(y.shape[-2], skip.shape[-2])
            y, skip = y[..., :h, :], skip[..., :h, :]

        out = y + skip
        out = out.permute(0, 2, 3, 1)
        out = self.norm(out)
        out = out.permute(0, 3, 1, 2)
        return F.relu(out)


# --------------------------------------------------------------------------- #
# 3. Grouped Spatial Module: one kernel per electrode group, then fusion
# --------------------------------------------------------------------------- #
class GroupedSpatialModule(nn.Module):
    """
    Block 1: one AsymmetricConvBlock per electrode group, kernel (|group|, 4),
             each with its own weights. Every group -> (B, f1, 1, T').
    Stack:   concatenate groups along height -> (B, f1, G, T').
    Block 2: AsymmetricConvBlock with kernel (G, 4) fusing all groups
             -> (B, f2, 1, T'').
    GAP ->   (B, f2)
    """

    def __init__(self, groups=ELECTRODE_GROUPS, flip_channels=FLIP_CHANNELS,
                 f1=16, f2=32, se_reduction=16):
        super().__init__()
        self.group_names = list(groups.keys())
        n_groups = len(self.group_names)

        # Channel indices per group, stored as buffers (move with .to(device)).
        for name in self.group_names:
            self.register_buffer(
                f"idx_{name}", torch.tensor(groups[name], dtype=torch.long),
                persistent=False,
            )
        self.flip_channels = list(flip_channels)

        # One independent asymmetric conv block per group.
        self.group_blocks = nn.ModuleDict({
            name: AsymmetricConvBlock(
                in_channels=1, out_channels=f1,
                kernel_size=(len(groups[name]), 4), dilation=(1, 2),
                se_reduction=se_reduction,
            )
            for name in self.group_names
        })

        # Fusion across the G groups.
        self.fusion_block = AsymmetricConvBlock(
            in_channels=f1, out_channels=f2,
            kernel_size=(n_groups, 4), dilation=(1, 2),
            se_reduction=se_reduction,
        )
        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        # x: (B, 1, C, T_segment)
        if self.flip_channels:
            x = x.clone()
            x[:, :, self.flip_channels, :] = -x[:, :, self.flip_channels, :]

        group_feats = []
        for name in self.group_names:
            idx = getattr(self, f"idx_{name}")
            xg = x.index_select(2, idx)                  # (B, 1, |g|, T)
            group_feats.append(self.group_blocks[name](xg))  # (B, f1, 1, T')

        x = torch.cat(group_feats, dim=2)                # (B, f1, G, T')
        x = self.fusion_block(x)                         # (B, f2, 1, T'')
        return self.gap(x).flatten(1)                    # (B, f2)


# --------------------------------------------------------------------------- #
# 4. Temporal Convolutional Network (unchanged)
# --------------------------------------------------------------------------- #
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation,
                 dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size,
                      padding=padding, dilation=dilation),
            Chomp1d(padding), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, kernel_size,
                      padding=padding, dilation=dilation),
            Chomp1d(padding), nn.ReLU(), nn.Dropout(dropout),
        )
        self.downsample = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else None
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)


class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=3, dropout=0.2):
        super().__init__()
        layers = []
        for i, out_ch in enumerate(num_channels):
            in_ch = num_inputs if i == 0 else num_channels[i - 1]
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size,
                                        2 ** i, dropout))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# --------------------------------------------------------------------------- #
# 5. Full model
# --------------------------------------------------------------------------- #
class AsymSETNetGrouped(nn.Module):
    """AsymSETNet with a separate spatial kernel per electrode group.

    Args:
        n_channels:   total electrode channels in the input (21 here).
        n_segments:   temporal segments per window; must divide T_total
                      (768 samples -> 3, 4, 6 or 8; NOT 5).
        groups:       dict name -> list of channel indices.
        flip_channels: channel indices to sign-flip before grouping.
    """

    def __init__(self, n_channels=21, n_segments=3, groups=ELECTRODE_GROUPS,
                 flip_channels=FLIP_CHANNELS, spatial_f1=16, spatial_f2=32,
                 tcn_channels=(32,8), se_reduction=16, dropout=0.2):
        super().__init__()
        used = sorted(i for g in groups.values() for i in g)
        assert max(used) < n_channels, "Group index exceeds n_channels"
        assert len(used) == len(set(used)), "A channel appears in two groups"

        self.n_segments = n_segments
        self.spatial_module = GroupedSpatialModule(
            groups=groups, flip_channels=flip_channels,
            f1=spatial_f1, f2=spatial_f2, se_reduction=se_reduction,
        )
        self.temporal_module = TemporalConvNet(
            num_inputs=spatial_f2, num_channels=list(tcn_channels),
            kernel_size=3, dropout=dropout,
        )
        self.classifier = nn.Linear(tcn_channels[-1], 1)

    def forward(self, x):
        """x: (B, C, T_total) -> logits (B,)"""
        b, c, t_total = x.shape
        n = self.n_segments
        assert t_total % n == 0, "T_total must be divisible by n_segments"
        t_seg = t_total // n

        x = x.view(b, c, n, t_seg).permute(0, 2, 1, 3).contiguous()
        x = x.view(b * n, 1, c, t_seg)                 # (B*N, 1, C, T_seg)

        feats = self.spatial_module(x)                 # (B*N, f2)
        feats = feats.view(b, n, -1).permute(0, 2, 1)  # (B, f2, N)

        temporal_out = self.temporal_module(feats)     # (B, C_tcn, N)
        pooled = temporal_out[:, :, -1]
        return self.classifier(pooled).squeeze(-1)

    def predict_proba(self, x):
        return torch.sigmoid(self.forward(x))


# --------------------------------------------------------------------------- #
# 6. Smoke test
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    torch.manual_seed(0)

    B, C, T_TOTAL = 8, 21, 3 * 256      # matches the processed windows (21, 768)
    N_SEGMENTS = 3                      # 768 / 3 = 256 samples (1 s) per segment

    model = AsymSETNetGrouped(n_channels=C, n_segments=N_SEGMENTS)
    x = torch.randn(B, C, T_TOTAL)
    y = torch.randint(0, 2, (B,)).float()

    logits = model(x)
    loss = nn.BCEWithLogitsLoss()(logits, y)
    loss.backward()

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Input shape:           {tuple(x.shape)}")
    print(f"Output (logits) shape: {tuple(logits.shape)}")
    print(f"BCE loss:              {loss.item():.4f}")
    print(f"Trainable parameters:  {n_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    optimizer.step()
    print("One optimizer step completed successfully.")

Input shape:           (8, 21, 768)
Output (logits) shape: (8,)
BCE loss:              0.7373
Trainable parameters:  25,329
One optimizer step completed successfully.


In [ ]:
"""
Train AsymSETNetGrouped on CHB-MIT with 10-fold patient-grouped cross-validation.

For each outer fold:
  1. The training patients are split again by patient into train / validation
     (validation is used for early stopping and threshold selection only).
  2. Per-channel normalisation stats are computed on the inner-train windows.
  3. The model is trained with class-weighted BCE and early stopping on val AUPRC.
  4. The best model (kept in memory) is evaluated once on the held-out test patients.

Nothing is written to disk.
"""
import copy
import time

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (average_precision_score, balanced_accuracy_score,
                             confusion_matrix, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedGroupKFold
# ------------------------------- configuration ------------------------------ #
CFG = dict(
    n_segments=3,          # 768 samples / 3 = 1 s per segment
    batch_size=256,
    max_epochs=50,
    patience=8,            # early stopping on val AUPRC
    lr=1e-3,
    weight_decay=1e-4,
    max_pos_weight=50.0,   # cap on BCE pos_weight for extreme imbalance
    grad_clip=1.0,
    seed=0,
    folds_to_run=None,     # e.g. [0, 1] for a quick run; None = all folds
)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"


# --------------------------------- helpers ---------------------------------- #
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def inner_train_val_split(idx, y_bin, groups, seed, n_splits=9):
    """Split outer-train indices by patient into inner train / validation."""
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_rel, va_rel in cv.split(np.zeros(len(idx)), y_bin[idx], groups[idx]):
        if y_bin[idx[va_rel]].any():          # make sure val contains seizures
            return idx[tr_rel], idx[va_rel]
    raise RuntimeError("Could not find a validation split containing positives.")


def channel_stats(X, idx, chunk=4096):
    """Per-channel mean/std over the given windows, computed in chunks."""
    C = X.shape[1]
    s = np.zeros(C, np.float64)
    s2 = np.zeros(C, np.float64)
    n = 0
    for i in range(0, len(idx), chunk):
        xb = X[np.sort(idx[i:i + chunk])].astype(np.float64)
        s += xb.sum(axis=(0, 2))
        s2 += (xb ** 2).sum(axis=(0, 2))
        n += xb.shape[0] * xb.shape[2]
    mean = s / n
    std = np.sqrt(np.maximum(s2 / n - mean ** 2, 1e-12))
    return (torch.tensor(mean, dtype=torch.float32, device=DEVICE).view(1, C, 1),
            torch.tensor(std, dtype=torch.float32, device=DEVICE).view(1, C, 1))


def iterate_batches(X, y, idx, batch_size, mean, std, shuffle, rng=None):
    """Yield normalised (x, y) tensor batches without copying the full split."""
    order = rng.permutation(idx) if shuffle else idx
    for i in range(0, len(order), batch_size):
        b = np.sort(order[i:i + batch_size])  # sorted indexing is faster on large arrays
        xb = torch.from_numpy(X[b]).to(DEVICE, non_blocking=True)
        yb = torch.from_numpy(y[b].astype(np.float32)).to(DEVICE, non_blocking=True)
        yield (xb - mean) / std, yb, b


@torch.no_grad()
def predict(model, X, y, idx, mean, std, batch_size):
    model.eval()
    probs = np.empty(len(idx), np.float32)
    pos = {j: k for k, j in enumerate(idx)}
    for xb, _, b in iterate_batches(X, y, idx, batch_size, mean, std, shuffle=False):
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            logits = model(xb)
        p = torch.sigmoid(logits.float()).cpu().numpy()
        probs[[pos[j] for j in b]] = p
    return probs


def best_threshold(y_true, probs):
    """Threshold maximising Youden's J (sensitivity + specificity - 1)."""
    fpr, tpr, thr = roc_curve(y_true, probs)
    return float(thr[np.argmax(tpr - fpr)])


def metrics(y_true, probs, thr):
    y_pred = (probs >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    has_both = len(np.unique(y_true)) == 2
    return dict(
        auroc=roc_auc_score(y_true, probs) if has_both else np.nan,
        auprc=average_precision_score(y_true, probs) if has_both else np.nan,
        sens=tp / (tp + fn) if tp + fn else np.nan,
        spec=tn / (tn + fp) if tn + fp else np.nan,
        bal_acc=balanced_accuracy_score(y_true, y_pred),
        thr=thr,
    )


# ------------------------------ training loop ------------------------------- #
def train_one_fold(k, X, y_bin, groups, tr_idx, te_idx):
    set_seed(CFG["seed"] + k)
    rng = np.random.default_rng(CFG["seed"] + k)

    in_tr, in_va = inner_train_val_split(tr_idx, y_bin, groups, CFG["seed"] + k)
    mean, std = channel_stats(X, in_tr)

    n_pos = y_bin[in_tr].sum()
    n_neg = len(in_tr) - n_pos
    pos_weight = min(n_neg / max(n_pos, 1), CFG["max_pos_weight"])

    model = AsymSETNetGrouped(n_channels=X.shape[1], n_segments=CFG["n_segments"]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"],
                                 weight_decay=CFG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max",
                                                           factor=0.5, patience=3)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    print(f"\n=== Fold {k} | train {len(in_tr)} (pos {n_pos}) | val {len(in_va)} "
          f"| test {len(te_idx)} | val patients {sorted(set(groups[in_va]))} "
          f"| pos_weight {pos_weight:.1f}")

    best_auprc, best_state, best_epoch, bad_epochs = -1.0, None, 0, 0
    for epoch in range(1, CFG["max_epochs"] + 1):
        model.train()
        t0, run_loss, n_seen = time.time(), 0.0, 0
        for xb, yb, _ in iterate_batches(X, y_bin, in_tr, CFG["batch_size"],
                                         mean, std, shuffle=True, rng=rng):
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=USE_AMP):
                logits = model(xb)
            loss = criterion(logits.float(), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.item() * len(yb)
            n_seen += len(yb)

        va_probs = predict(model, X, y_bin, in_va, mean, std, CFG["batch_size"])
        va_auprc = average_precision_score(y_bin[in_va], va_probs)
        va_auroc = roc_auc_score(y_bin[in_va], va_probs)
        scheduler.step(va_auprc)

        improved = va_auprc > best_auprc
        if improved:
            best_auprc, best_epoch, bad_epochs = va_auprc, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad_epochs += 1

        print(f"  ep {epoch:3d} | loss {run_loss / n_seen:.4f} | val AUROC {va_auroc:.4f} "
              f"| val AUPRC {va_auprc:.4f} | lr {optimizer.param_groups[0]['lr']:.1e} "
              f"| {time.time() - t0:.0f}s{' *' if improved else ''}")
        if bad_epochs >= CFG["patience"]:
            print(f"  early stop (best epoch {best_epoch})")
            break

    model.load_state_dict(best_state)
    va_probs = predict(model, X, y_bin, in_va, mean, std, CFG["batch_size"])
    thr = best_threshold(y_bin[in_va], va_probs)

    te_probs = predict(model, X, y_bin, te_idx, mean, std, CFG["batch_size"])
    m = metrics(y_bin[te_idx], te_probs, thr)
    print(f"  TEST {sorted(set(groups[te_idx]))} | AUROC {m['auroc']:.4f} | "
          f"AUPRC {m['auprc']:.4f} | sens {m['sens']:.3f} | spec {m['spec']:.3f} | "
          f"bal_acc {m['bal_acc']:.3f} | thr {thr:.3f}")
    return m, te_probs, thr


def main():
    print(f"Device: {DEVICE}")
    X, y, patient = load_dataset()
    y_bin = (np.asarray(y) > 0).astype(np.int64)
    groups = make_groups(patient)

    folds = get_cv_splits(y, patient)
    if CFG["folds_to_run"] is not None:
        folds = [f for f in folds if f[0] in CFG["folds_to_run"]]

    oof_probs = np.full(len(y_bin), np.nan, np.float32)
    oof_pred = np.full(len(y_bin), -1, np.int8)
    results = []
    for k, tr, te in folds:
        m, te_probs, thr = train_one_fold(k, X, y_bin, groups, tr, te)
        oof_probs[te] = te_probs
        oof_pred[te] = (te_probs >= thr).astype(np.int8)
        results.append(m)

    print("\n=========== Cross-validation summary ===========")
    for key in ["auroc", "auprc", "sens", "spec", "bal_acc"]:
        vals = np.array([r[key] for r in results], dtype=float)
        print(f"{key:>8}: {np.nanmean(vals):.4f} ± {np.nanstd(vals):.4f}")

    done = oof_pred >= 0
    yt, yp = y_bin[done], oof_pred[done]
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    print(f"\nPooled out-of-fold: AUROC {roc_auc_score(yt, oof_probs[done]):.4f} | "
          f"AUPRC {average_precision_score(yt, oof_probs[done]):.4f} | "
          f"sens {tp / (tp + fn):.3f} | spec {tn / (tn + fp):.3f}")
    return results, oof_probs


results, oof_probs = main()

Device: cuda
fold   n_train   n_test test_pos%  test patients
   0      6490      718    50.00%  [np.str_('0'), np.str_('21'), np.str_('9')]
   1      6344      864    50.00%  [np.str_('10'), np.str_('23')]
   2      6202     1006    50.00%  [np.str_('11'), np.str_('4')]
   3      6794      414    50.00%  [np.str_('12'), np.str_('20')]
   4      6614      594    50.00%  [np.str_('13'), np.str_('22'), np.str_('6')]
   5      5024     2184    50.00%  [np.str_('14'), np.str_('2'), np.str_('7')]
   6      6700      508    50.00%  [np.str_('15'), np.str_('19'), np.str_('5'), np.str_('8')]
   7      7014      194    50.00%  [np.str_('16')]
   8      6846      362    50.00%  [np.str_('17'), np.str_('18')]
   9      6844      364    50.00%  [np.str_('1'), np.str_('3')]

Overall positive rate: 50.00%  |  windows: 7208  |  groups: 24


/tmp/ipykernel_1844908/1982663704.py:133: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)



=== Fold 0 | train 6246 (pos 3123) | val 244 | test 718 | val patients [np.str_('1'), np.str_('20')] | pos_weight 1.0
  ep   1 | loss 0.6445 | val AUROC 0.8839 | val AUPRC 0.8830 | lr 1.0e-03 | 1s *
  ep   2 | loss 0.5844 | val AUROC 0.9019 | val AUPRC 0.8960 | lr 1.0e-03 | 0s *
  ep   3 | loss 0.5592 | val AUROC 0.9348 | val AUPRC 0.9324 | lr 1.0e-03 | 0s *
  ep   4 | loss 0.5438 | val AUROC 0.9481 | val AUPRC 0.9430 | lr 1.0e-03 | 0s *
  ep   5 | loss 0.5176 | val AUROC 0.9532 | val AUPRC 0.9420 | lr 1.0e-03 | 0s
  ep   6 | loss 0.4817 | val AUROC 0.9519 | val AUPRC 0.9306 | lr 1.0e-03 | 1s
  ep   7 | loss 0.4165 | val AUROC 0.9506 | val AUPRC 0.9171 | lr 1.0e-03 | 0s
  ep   8 | loss 0.3693 | val AUROC 0.9547 | val AUPRC 0.9299 | lr 5.0e-04 | 0s
  ep   9 | loss 0.3304 | val AUROC 0.9524 | val AUPRC 0.9181 | lr 5.0e-04 | 0s
  ep  10 | loss 0.3398 | val AUROC 0.9570 | val AUPRC 0.9312 | lr 5.0e-04 | 0s
  ep  11 | loss 0.3362 | val AUROC 0.9529 | val AUPRC 0.9220 | lr 5.0e-04 | 0s
  ep

/tmp/ipykernel_1844908/1982663704.py:133: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)



=== Fold 1 | train 6052 (pos 3026) | val 292 | test 864 | val patients [np.str_('0')] | pos_weight 1.0
  ep   1 | loss 0.6395 | val AUROC 0.9881 | val AUPRC 0.9647 | lr 1.0e-03 | 0s *
  ep   2 | loss 0.5939 | val AUROC 0.9890 | val AUPRC 0.9688 | lr 1.0e-03 | 0s *
  ep   3 | loss 0.5657 | val AUROC 0.9934 | val AUPRC 0.9929 | lr 1.0e-03 | 0s *
  ep   4 | loss 0.5372 | val AUROC 0.9972 | val AUPRC 0.9975 | lr 1.0e-03 | 0s *
  ep   5 | loss 0.5155 | val AUROC 0.9956 | val AUPRC 0.9958 | lr 1.0e-03 | 0s
  ep   6 | loss 0.4846 | val AUROC 0.9952 | val AUPRC 0.9951 | lr 1.0e-03 | 0s
  ep   7 | loss 0.4367 | val AUROC 0.9956 | val AUPRC 0.9956 | lr 1.0e-03 | 0s
  ep   8 | loss 0.3547 | val AUROC 0.9937 | val AUPRC 0.9925 | lr 5.0e-04 | 0s
  ep   9 | loss 0.3311 | val AUROC 0.9932 | val AUPRC 0.9920 | lr 5.0e-04 | 0s
  ep  10 | loss 0.3131 | val AUROC 0.9936 | val AUPRC 0.9922 | lr 5.0e-04 | 1s
  ep  11 | loss 0.2980 | val AUROC 0.9925 | val AUPRC 0.9909 | lr 5.0e-04 | 0s
  ep  12 | loss 0.2

...


=== Fold 7 | train 6744 (pos 3372) | val 270 | test 194 | val patients [np.str_('1'), np.str_('18')] | pos_weight 1.0
  ep   1 | loss 0.6276 | val AUROC 0.9541 | val AUPRC 0.9615 | lr 1.0e-03 | 0s *
  ep   2 | loss 0.5652 | val AUROC 0.9588 | val AUPRC 0.9651 | lr 1.0e-03 | 0s *
  ep   3 | loss 0.5484 | val AUROC 0.9622 | val AUPRC 0.9674 | lr 1.0e-03 | 0s *
  ep   4 | loss 0.5354 | val AUROC 0.9646 | val AUPRC 0.9689 | lr 1.0e-03 | 0s *
  ep   5 | loss 0.5112 | val AUROC 0.9555 | val AUPRC 0.9595 | lr 1.0e-03 | 0s
  ep   6 | loss 0.5031 | val AUROC 0.9511 | val AUPRC 0.9584 | lr 1.0e-03 | 0s
  ep   7 | loss 0.4982 | val AUROC 0.9485 | val AUPRC 0.9570 | lr 1.0e-03 | 1s
  ep   8 | loss 0.4760 | val AUROC 0.9450 | val AUPRC 0.9531 | lr 5.0e-04 | 1s
  ep   9 | loss 0.4503 | val AUROC 0.9354 | val AUPRC 0.9444 | lr 5.0e-04 | 0s
  ep  10 | loss 0.4371 | val AUROC 0.9578 | val AUPRC 0.9634 | lr 5.0e-04 | 0s
  ep  11 | loss 0.4024 | val AUROC 0.9507 | val AUPRC 0.9570 | lr 5.0e-04 | 0s
  ep

/tmp/ipykernel_1844908/1982663704.py:133: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)



=== Fold 8 | train 6266 (pos 3133) | val 580 | test 362 | val patients [np.str_('1'), np.str_('19'), np.str_('22')] | pos_weight 1.0
  ep   1 | loss 0.6758 | val AUROC 0.9415 | val AUPRC 0.9376 | lr 1.0e-03 | 0s *
  ep   2 | loss 0.5961 | val AUROC 0.9419 | val AUPRC 0.9404 | lr 1.0e-03 | 0s *
  ep   3 | loss 0.5565 | val AUROC 0.9446 | val AUPRC 0.9476 | lr 1.0e-03 | 0s *
  ep   4 | loss 0.5421 | val AUROC 0.9485 | val AUPRC 0.9531 | lr 1.0e-03 | 0s *
  ep   5 | loss 0.5225 | val AUROC 0.9449 | val AUPRC 0.9510 | lr 1.0e-03 | 0s
  ep   6 | loss 0.5061 | val AUROC 0.9375 | val AUPRC 0.9449 | lr 1.0e-03 | 0s
  ep   7 | loss 0.4960 | val AUROC 0.9277 | val AUPRC 0.9388 | lr 1.0e-03 | 0s
  ep   8 | loss 0.4854 | val AUROC 0.9156 | val AUPRC 0.9309 | lr 5.0e-04 | 0s
  ep   9 | loss 0.4688 | val AUROC 0.8856 | val AUPRC 0.9157 | lr 5.0e-04 | 0s
  ep  10 | loss 0.4551 | val AUROC 0.8640 | val AUPRC 0.9032 | lr 5.0e-04 | 1s
  ep  11 | loss 0.4536 | val AUROC 0.8673 | val AUPRC 0.9009 | lr 5.

/tmp/ipykernel_1844908/1982663704.py:133: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)



=== Fold 9 | train 5678 (pos 2839) | val 1166 | test 364 | val patients [np.str_('10'), np.str_('2'), np.str_('4')] | pos_weight 1.0
  ep   1 | loss 0.6619 | val AUROC 0.9402 | val AUPRC 0.9291 | lr 1.0e-03 | 0s *
  ep   2 | loss 0.6089 | val AUROC 0.9425 | val AUPRC 0.9310 | lr 1.0e-03 | 0s *
  ep   3 | loss 0.5725 | val AUROC 0.9388 | val AUPRC 0.9256 | lr 1.0e-03 | 0s
  ep   4 | loss 0.5515 | val AUROC 0.9240 | val AUPRC 0.9102 | lr 1.0e-03 | 0s
  ep   5 | loss 0.5239 | val AUROC 0.9019 | val AUPRC 0.8912 | lr 1.0e-03 | 0s
  ep   6 | loss 0.4794 | val AUROC 0.8824 | val AUPRC 0.8782 | lr 5.0e-04 | 0s
  ep   7 | loss 0.4174 | val AUROC 0.8918 | val AUPRC 0.8878 | lr 5.0e-04 | 0s
  ep   8 | loss 0.3934 | val AUROC 0.9036 | val AUPRC 0.8979 | lr 5.0e-04 | 0s
  ep   9 | loss 0.3565 | val AUROC 0.8927 | val AUPRC 0.8901 | lr 5.0e-04 | 0s
  ep  10 | loss 0.3408 | val AUROC 0.8809 | val AUPRC 0.8853 | lr 2.5e-04 | 0s
  early stop (best epoch 2)
  TEST [np.str_('1'), np.str_('3')] | AUROC 